
# TP : Prédire le prix de location de logements à Antananarivo avec une régression linéaire multiple

## 🎯 Objectifs pédagogiques

- Appliquer un pipeline de prétraitement complet sur un jeu de données semi-structuré.
- Gérer les variables qualitatives, les valeurs manquantes, la multicolinéarité et la scalabilité.
- Construire, tester et évaluer un modèle de régression linéaire multiple.
- Déployer le modèle dans une application Python Streamlit avec interface utilisateur.

## 🗂️ Jeu de données

Le jeu de données doit être collecté ou scrappé dans les pages comme Facebook. Il doit comporter les colonnes suivantes :

- `quartier` (catégorielle)
- `superficie` (numérique)
- `nombre_chambres` (numérique)
- `douche_wc`(interieur ou exterieur)
- `type_d_acces` (sans, moto, voiture, voiture_avec_par_parking)
- `meublé` (booléen:  oui ou non)
- `état_général` (catégorielle : bon, moyen, mauvais)
- `loyer_mensuel` (target)

## 🧪 Étapes du TP

### 📌 Partie 1 : Préparation des données
- Lecture du dataset brut
- Gestion des valeurs manquantes
- Encodage des variables catégorielles
- Création de variables dérivées
- Détection et suppression des variables fortement corrélées
- Standardisation et normalisation

### 📌 Partie 2 : Modélisation

- Séparation train/test
- Implémentation de la régression linéaire multiple
- Évaluation : R², RMSE
- Vérification des hypothèses d'élligibilité de la régression linéaire multiple (surtout sur les erreurs)

### 📌 Partie 3 : Optimisation du modèle

- Sélection de variables : backward elimination, RFE (à documenter)

### 📌 Partie 4 : Déploiement d’une application Streamlit

- Interface de saisie utilisateur
- Affichage du loyer prédit
- Visualisation des poids des variables
- Affichage sur la carte interactive

## 🧭 Carte interactive (option avancée)

Utiliser `streamlit-folium` pour permettre à l’utilisateur de cliquer sur une carte et de récupérer les coordonnées GPS. À partir de ces coordonnées, déterminer automatiquement le quartier en utilisant un fichier GeoJSON ou un système de polygones avec `geopandas`.

## 🔧 Technologies à utiliser

- `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, `joblib`
- `streamlit`, `folium`, `streamlit-folium`
- Optionnel : `geopandas`, `shapely`

## 💡 Bonus

- Carte interactive avec folium
- Simuler des données additionnelles (pollution, sécurité)
- Tri automatique des caractéristiques influentes




In [63]:
# PARTIE 1

In [64]:
import pandas as pd

# Charger le jeu de données
df = pd.read_csv('mock_logements_antananarivo_1000.csv')
print("hello")
df.head()

hello


,quartier,superficie,nombre_chambres,douche_wc,type_d_acces,meuble,etat_general,loyer_mensuel
0,Andavamamba,46,2,exterieur,voiture,non,bon,515000
1,Itaosy,47,1,interieur,moto,non,bon,691000
2,Ambatoroka,15,2,interieur,voiture_avec_parking,oui,bon,1112000
3,Ivandry,97,1,interieur,moto,oui,mauvais,3009000
4,Ankatso,56,2,interieur,moto,non,moyen,825000


In [65]:
# Somme des valeurs manquantes par colonne
df.isnull().sum()

quartier           0
superficie         0
nombre_chambres    0
douche_wc          0
type_d_acces       0
meuble             0
etat_general       0
loyer_mensuel      0
dtype: int64

In [66]:
df['douche_wc'] = df['douche_wc'].map({'exterieur': 0, 'interieur': 1})
df['meuble'] = df['meuble'].map({'non': 0, 'oui': 1})
df['etat_general'] = df['etat_general'].map({'bon': 10, 'moyen': 5, 'mauvais': 0})
df['type_d_acces'] = df['type_d_acces'].map({'sans': 0, 'moto': 5, 'voiture': 10, 'voiture_avec_parking': 15})
df = pd.get_dummies(df, columns=['quartier'], dtype=int)
df.head()

,superficie,nombre_chambres,douche_wc,type_d_acces,meuble,etat_general,loyer_mensuel,quartier_Alasora,quartier_Ambatoroka,quartier_Analakely,quartier_Andavamamba,quartier_Ankatso,quartier_Ankorondrano,quartier_Isoraka,quartier_Itaosy,quartier_Ivandry,quartier_Mahamasina
0,46,2,0,10,0,10,515000,0,0,0,1,0,0,0,0,0,0
1,47,1,1,5,0,10,691000,0,0,0,0,0,0,0,1,0,0
2,15,2,1,15,1,10,1112000,0,1,0,0,0,0,0,0,0,0
3,97,1,1,5,1,0,3009000,0,0,0,0,0,0,0,0,1,0
4,56,2,1,5,0,5,825000,0,0,0,0,1,0,0,0,0,0


In [67]:
#Création de variables dérivées
df["score_confort"] = (
    (df["douche_wc"] == 1).astype(int)
    + (df["meuble"] == 1).astype(int)
    + (df["type_d_acces"] >= 10).astype(int)
    + (df["etat_general"] >= 10).astype(int)
)

df.head()


,superficie,nombre_chambres,douche_wc,type_d_acces,meuble,etat_general,loyer_mensuel,quartier_Alasora,quartier_Ambatoroka,quartier_Analakely,quartier_Andavamamba,quartier_Ankatso,quartier_Ankorondrano,quartier_Isoraka,quartier_Itaosy,quartier_Ivandry,quartier_Mahamasina,score_confort
0,46,2,0,10,0,10,515000,0,0,0,1,0,0,0,0,0,0,2
1,47,1,1,5,0,10,691000,0,0,0,0,0,0,0,1,0,0,2
2,15,2,1,15,1,10,1112000,0,1,0,0,0,0,0,0,0,0,4
3,97,1,1,5,1,0,3009000,0,0,0,0,0,0,0,0,1,0,2
4,56,2,1,5,0,5,825000,0,0,0,0,1,0,0,0,0,0,1


In [68]:
corr = df.corr(numeric_only=True)
print(corr)

                       superficie  nombre_chambres  douche_wc  type_d_acces  \
superficie               1.000000        -0.021080  -0.009728      0.067394   
nombre_chambres         -0.021080         1.000000   0.019514     -0.005192   
douche_wc               -0.009728         0.019514   1.000000     -0.011192   
type_d_acces             0.067394        -0.005192  -0.011192      1.000000   
meuble                   0.008441        -0.024998   0.070024     -0.009599   
etat_general            -0.012597        -0.070751   0.002929      0.003864   
loyer_mensuel            0.360734         0.121309   0.112148      0.243085   
quartier_Alasora        -0.012742        -0.025377  -0.033674      0.043842   
quartier_Ambatoroka      0.022283         0.012166  -0.027902      0.012528   
quartier_Analakely      -0.031020        -0.029683   0.051120     -0.002136   
quartier_Andavamamba     0.027773         0.034875   0.005166      0.006283   
quartier_Ankatso         0.025396        -0.004672  

In [69]:

threshold = 0.9

# variables à supprimer
to_drop = set()

for i in range(len(corr.columns)):
    for j in range(i):
        if abs(corr.iloc[i, j]) > threshold:
            colname = corr.columns[i]
            to_drop.add(colname)

df_reduced = df.drop(columns=to_drop)
df_reduced.head()

,superficie,nombre_chambres,douche_wc,type_d_acces,meuble,etat_general,loyer_mensuel,quartier_Alasora,quartier_Ambatoroka,quartier_Analakely,quartier_Andavamamba,quartier_Ankatso,quartier_Ankorondrano,quartier_Isoraka,quartier_Itaosy,quartier_Ivandry,quartier_Mahamasina,score_confort
0,46,2,0,10,0,10,515000,0,0,0,1,0,0,0,0,0,0,2
1,47,1,1,5,0,10,691000,0,0,0,0,0,0,0,1,0,0,2
2,15,2,1,15,1,10,1112000,0,1,0,0,0,0,0,0,0,0,4
3,97,1,1,5,1,0,3009000,0,0,0,0,0,0,0,0,1,0,2
4,56,2,1,5,0,5,825000,0,0,0,0,1,0,0,0,0,0,1


In [70]:
X = df_reduced.drop("loyer_mensuel", axis=1)
y = df_reduced["loyer_mensuel"]

In [71]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [72]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(X_scaled.head())

   superficie  nombre_chambres  douche_wc  type_d_acces    meuble  \
0   -0.527885         0.131760  -1.736682      0.353122 -0.635999   
1   -0.487234        -1.013981   0.575811     -0.740136 -0.635999   
2   -1.788043         0.131760   0.575811      1.446380  1.572330   
3    1.545280        -1.013981   0.575811     -0.740136  1.572330   
4   -0.121382         0.131760   0.575811     -0.740136 -0.635999   

   etat_general  quartier_Alasora  quartier_Ambatoroka  quartier_Analakely  \
0      0.921328         -0.331478            -0.396746           -0.304789   
1      0.921328         -0.331478            -0.396746           -0.304789   
2      0.921328         -0.331478             2.520504           -0.304789   
3     -1.841274         -0.331478            -0.396746           -0.304789   
4     -0.459973         -0.331478            -0.396746           -0.304789   

   quartier_Andavamamba  quartier_Ankatso  quartier_Ankorondrano  \
0              3.086473         -0.322107       